In [6]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"I am using: {device}")

I am using: cuda


In [8]:
import pandas as pd
import re
import torch
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [9]:
%pip install emoji

Note: you may need to restart the kernel to use updated packages.


In [10]:
import pandas as pd

# قراءة البيانات
train_df = pd.read_csv('data/Train.csv')
test_df = pd.read_csv('data/Test.csv')

# قراءة وتجهيز قاموس الإيموجيز
mapping_df = pd.read_csv('data/Mapping.csv')
label_mapping = dict(zip(mapping_df['emoticons'], mapping_df['number']))


In [11]:
import pandas as pd
import re
import emoji
import numpy as np

def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    # Remove @user mentions
    text = re.sub(r'@user', '', text)
    
    # Remove URLs
    text = re.sub(r'http\S+', '', text)
    
    # Remove hashtags symbol but keep the word
    text = re.sub(r'#(\w+)', r'\1', text)
    
    # Remove emojis to prevent data leakage (so the model learns words, not shapes)
    text = emoji.replace_emoji(text, replace='')
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

print("Loading data...")
df = pd.read_csv('data/Train.csv')
print(f"Original shape: {df.shape}")

print("Cleaning text...")
df['TEXT'] = df['TEXT'].apply(clean_text)

# Drop empty texts
df = df[df['TEXT'] != ""]

print("Applying undersampling to majority classes...")
MAX_SAMPLES = 3000
balanced_dfs = []

for label in df['Label'].unique():
    subset = df[df['Label'] == label]
    if len(subset) > MAX_SAMPLES:
        # Undersample randomly
        subset = subset.sample(n=MAX_SAMPLES, random_state=42)
    balanced_dfs.append(subset)

balanced_df = pd.concat(balanced_dfs).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"New shape after undersampling: {balanced_df.shape}")
print("New class distribution:")
print(balanced_df['Label'].value_counts())

balanced_df.to_csv('data/Clean_Train.csv', index=False)
print("Saved cleaned and balanced data to data/Clean_Train.csv")


Loading data...
Original shape: (70000, 3)
Cleaning text...
Applying undersampling to majority classes...
New shape after undersampling: (47535, 3)
New class distribution:
Label
13    3000
9     3000
2     3000
7     3000
15    3000
3     3000
16    2832
17    2751
1     2592
11    2434
14    2342
6     2083
8     1992
5     1977
12    1909
4     1878
19    1745
18    1722
10    1721
0     1557
Name: count, dtype: int64
Saved cleaned and balanced data to data/Clean_Train.csv


In [12]:
import pandas as pd
import numpy as np
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

print("Loading cleaned data...")
df = pd.read_csv('data/Clean_Train.csv')
df = df.dropna(subset=['TEXT', 'Label'])

X = df['TEXT']
y = df['Label']

print("Splitting data...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("TF-IDF Vectorization...")
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Training Logistic Regression with class_weight='balanced'...")
model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42, n_jobs=-1)
model.fit(X_train_vec, y_train)

print("Evaluating...")
y_pred = model.predict(X_test_vec)
f1 = f1_score(y_test, y_pred, average='weighted')
print(f"Weighted F1-Score: {f1:.4f}")
print("Classification Report:")
print(classification_report(y_test, y_pred))

import os

print("Saving models...")
os.makedirs('models', exist_ok=True)
with open('models/logistic_model.pkl', 'wb') as f:
    pickle.dump(model, f)
    
with open('models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

# Recreate reverse_label_map from classes
reverse_label_map = {i: i for i in range(20)}
with open('models/reverse_label_map.pkl', 'wb') as f:
    pickle.dump(reverse_label_map, f)

print("Done!")


Loading cleaned data...
Splitting data...
TF-IDF Vectorization...
Training Logistic Regression with class_weight='balanced'...


c:\Users\user\miniconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Evaluating...
Weighted F1-Score: 0.2323
Classification Report:
              precision    recall  f1-score   support

           0       0.07      0.09      0.08       311
           1       0.18      0.17      0.17       518
           2       0.22      0.17      0.19       600
           3       0.27      0.28      0.27       600
           4       0.09      0.12      0.10       376
           5       0.70      0.71      0.70       396
           6       0.15      0.18      0.16       417
           7       0.39      0.38      0.38       600
           8       0.17      0.23      0.19       398
           9       0.18      0.10      0.13       600
          10       0.10      0.12      0.11       344
          11       0.52      0.57      0.54       487
          12       0.32      0.52      0.39       382
          13       0.24      0.21      0.23       600
          14       0.20      0.18      0.19       468
          15       0.19      0.14      0.17       600
          16      

In [13]:
import os
import pandas as pd
import torch
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

print("Checking CUDA...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. تحميل الداتا النظيفة والموزونة اللي جهزتها
print("Loading cleaned data...")
df = pd.read_csv('data/Clean_Train.csv')
df = df.dropna(subset=['TEXT', 'Label'])

X = df['TEXT'].tolist()
y = df['Label'].tolist()

# تقسيم الداتا مع الحفاظ على التوزيع العادل
print("Splitting data...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. حساب أوزان الفئات وتمريرها للـ GPU
print("Computing class weights...")
classes = np.unique(y_train)
class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

# 3. الـ Tokenization المباشر والسريع
print("Loading tokenizer & Tokenizing...")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_encodings = tokenizer(X_train, padding="max_length", truncation=True, max_length=64, return_tensors="pt")
test_encodings = tokenizer(X_test, padding="max_length", truncation=True, max_length=64, return_tensors="pt")

# بناء الـ Dataset الكلاسيكي لـ PyTorch
class EmojiDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = EmojiDataset(train_encodings, y_train)
test_dataset = EmojiDataset(test_encodings, y_test)

# إنشاء الـ DataLoaders لإطعام كرت الشاشة بالـ Batches
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

# 4. تهيئة الموديل ودالة الخسارة (CrossEntropyLoss) بالأوزان
print("Loading BERT model...")
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=len(classes)).to(device)
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)

# 5. حلقة التدريب اليدوية (Native Training Loop) - أسرع وبدون تعقيدات
epochs = 1  # جرب epoch واحد كـ Test للتأكد، وبعدها بتقدر ترفعها لـ 3
print("Starting native PyTorch training loop...")

for epoch in range(epochs):
    model.train()
    total_loss = 0
    step = 0
    for batch in train_loader:
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fct(outputs.logits, labels)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        step += 1
        
        if step % 100 == 0:
            print(f"Epoch {epoch+1} - Step {step}/{len(train_loader)} - Loss: {loss.item():.4f}")
            
    print(f"--> Epoch {epoch+1}/{epochs} Finished - Average Loss: {total_loss/len(train_loader):.4f}")

# 6. التقييم النهائي وحساب المقاييس النظيفة
print("Evaluating BERT model...")
model.eval()
bert_preds = []
bert_targets = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        batch_preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        
        bert_preds.extend(batch_preds)
        bert_targets.extend(labels.cpu().numpy())

print("\n--- Final Clean BERT Results ---")
print("BERT Accuracy:", accuracy_score(bert_targets, bert_preds))
print("BERT Precision:", precision_score(bert_targets, bert_preds, average='weighted', zero_division=0))
print("BERT Recall:", recall_score(bert_targets, bert_preds, average='weighted', zero_division=0))
print("BERT F1-Score:", f1_score(bert_targets, bert_preds, average='weighted', zero_division=0))

# 7. حفظ الموديل والـ Tokenizer ليكونوا جاهزين للـ GUI
print("Saving model...")
os.makedirs('models/bert_model', exist_ok=True)
model.save_pretrained('models/bert_model')
tokenizer.save_pretrained('models/bert_model')

print("Done!")

Checking CUDA...
Using device: cuda
Loading cleaned data...
Splitting data...
Computing class weights...
Loading tokenizer & Tokenizing...


Loading BERT model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6180.65it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

Starting native PyTorch training loop...
Epoch 1 - Step 100/2377 - Loss: 3.0395
Epoch 1 - Step 200/2377 - Loss: 2.5897
Epoch 1 - Step 300/2377 - Loss: 2.8188
Epoch 1 - Step 400/2377 - Loss: 2.4944
Epoch 1 - Step 500/2377 - Loss: 2.7210
Epoch 1 - Step 600/2377 - Loss: 2.5807
Epoch 1 - Step 700/2377 - Loss: 2.4516
Epoch 1 - Step 800/2377 - Loss: 2.3244
Epoch 1 - Step 900/2377 - Loss: 2.2494
Epoch 1 - Step 1000/2377 - Loss: 2.5413
Epoch 1 - Step 1100/2377 - Loss: 2.3789
Epoch 1 - Step 1200/2377 - Loss: 2.4905
Epoch 1 - Step 1300/2377 - Loss: 1.9844
Epoch 1 - Step 1400/2377 - Loss: 2.8040
Epoch 1 - Step 1500/2377 - Loss: 2.3243
Epoch 1 - Step 1600/2377 - Loss: 1.9046
Epoch 1 - Step 1700/2377 - Loss: 2.3013
Epoch 1 - Step 1800/2377 - Loss: 2.1136
Epoch 1 - Step 1900/2377 - Loss: 2.6961
Epoch 1 - Step 2000/2377 - Loss: 1.8659
Epoch 1 - Step 2100/2377 - Loss: 2.0317
Epoch 1 - Step 2200/2377 - Loss: 2.3705
Epoch 1 - Step 2300/2377 - Loss: 2.0695
--> Epoch 1/1 Finished - Average Loss: 2.4884
Ev

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.35it/s]

Done!


In [18]:
import os
import pickle

os.makedirs('models', exist_ok=True)

with open('models/logistic_model.pkl', 'wb') as f:
    pickle.dump(lr_model, f)

with open('models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

reverse_label_map = {idx: label for label, idx in label_mapping.items()}
with open('models/reverse_label_map.pkl', 'wb') as f:
    pickle.dump(reverse_label_map, f)

bert_save_path = os.path.join('models', 'bert_model')
os.makedirs(bert_save_path, exist_ok=True)
model.save_pretrained(bert_save_path)
tokenizer.save_pretrained(bert_save_path)

print("All models saved successfully.")